# Trial — brapi-py

Interactive notebook for exploring the BrAPI **Trial** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Install / upgrade brapi-py — uncomment ONE of the options below:

# Option 1: install from PyPI (once the package is published)
# %pip install -q brapi-py

# Option 2: install from local source (for development / testing)
# import subprocess, sys, pathlib
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e',
#                        str(pathlib.Path('..').resolve())])

import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient
from dotenv import load_dotenv
import os

# Loads credentials from ../.env (never committed to git)
# Create a .env file in the project root with these keys:
#   BRAPI_BASE_URL=https://brapi.example.com
#   BRAPI_TOKEN_ENDPOINT=https://auth.example.com/token
#   BRAPI_CLIENT_ID=my-client
#   BRAPI_CLIENT_SECRET=my-secret
load_dotenv(dotenv_path='../.env')

client = BrapiClient(
    base_url=os.environ['BRAPI_BASE_URL'],
    token_endpoint=os.environ['BRAPI_TOKEN_ENDPOINT'],
    client_id=os.environ['BRAPI_CLIENT_ID'],
    client_secret=os.environ['BRAPI_CLIENT_SECRET'],
)
print('Client ready →', client)

## 2 — Search  (`POST /search/trials`)

Full-featured search. Handles both synchronous (HTTP 200) and asynchronous (HTTP 202 + polling) responses.

In [ ]:
# Basic search — filtered by common_crop_names()
df = (
    client.trial
    .common_crop_names(['Tomatillo'])
    .search()
    .to_df()
)
print(f'{len(df)} records returned')
df.head()

In [ ]:
# Immutable builder — fork the same base query safely
base = (
    client.trial
    .common_crop_names(['Tomatillo'])
)

q1_df = base.by_location_dbid(['b28911cf']).search().to_df()
q2_df = base.by_location_name(['Location Alpha']).search().to_df()

print('q1:', len(q1_df), '  q2:', len(q2_df))

In [ ]:
# Bulk filter — apply multiple criteria in one call
df = (
    client.trial
    .filter(
        common_crop_names=['Tomatillo'],
        location_dbids=['b28911cf'],
        location_names=['Location Alpha'],  # add more filters above
    )
    .search()
    .to_df()
)
df.head(10)

## 3 — List  (`GET /trials`)

Simpler GET endpoint — same filter state, mapped to query-string params.
Useful when the server does not support the search endpoint, or for quick lookups.

In [ ]:
# GET /trials — list records
df = (
    client.trial
    .common_crop_names(['Tomatillo'])
    .list()
    .to_df()
)
print(f'{len(df)} records')
df.head()

## 4 — Single-record CRUD

Get, create, update, and delete individual records.

In [ ]:
# GET /trials/ {id}  — fetch one record by primary key
TRIAL_DBID = 'example-001'   # ← change to a real ID

record = client.trial.get_by_id(TRIAL_DBID)
print(record)

In [ ]:
from brapi.entities.generated_trial import Trial

# POST /trials — create a new record
new_trial = Trial(
    trialDbId='',  # assigned by server
    trialName='Example Trial',
)
created = client.trial.create(new_trial)
print('Created trialDbId:', created.trialDbId)

In [ ]:
# PUT /trials/ {id} — update the record created above
# (run the create cell first to populate 'created')
created.active = 'updated value'
updated = client.trial.update(created.trialDbId, created)
print('Updated:', updated.trialDbId)

## 5 — Pipe transforms

`.pipe()` applies a pure function lazily — only one HTTP call is made regardless of how many transform stages are chained.

In [ ]:
def filter_non_null_ids(items):
    """Keep only records with a non-empty primary key."""
    return [r for r in items if r.trialDbId]

def add_label(items):
    """Attach a short display label to each record."""
    for r in items:
        r.label = f'Trial {r.trialDbId}'
    return items

df = (
    client.trial
    .common_crop_names(['Tomatillo'])
    .search()
    .pipe(filter_non_null_ids)
    .pipe(add_label)
    .to_df()
)
df.head(10)

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.trial
    .common_crop_names(['Tomatillo'])
    .search()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'active'
if 'active' in df.columns:
    df['active'].value_counts().head(10)
else:
    print('Column not present in this result set')

In [ ]:
import json, pandas as pd

# Explode the 'contacts' JSON column into separate rows
col = 'contacts'
if col in df.columns:
    id_col = 'trialDbId'
    exploded = (
        df.filter(items=[id_col, col])
        .dropna(subset=[col])
        .assign(**{col: lambda d: d[col].apply(json.loads)})
        .explode(col)
    )
    exploded.head(10)
else:
    print(f'Column {col!r} not present in this result set')

## 7 — Error handling

In [ ]:
import httpx

# 404 — record not found
try:
    client.trial.get_by_id('does-not-exist-xyz')
except httpx.HTTPStatusError as e:
    print(f'HTTP {e.response.status_code}: {e.request.url}')

# Async search poll timeout
try:
    client.trial.search(max_poll_attempts=1).to_list()
except TimeoutError as e:
    print('TimeoutError:', e)

# ValueError from create_many with only one item
try:
    client.trial.create_many([])
except (ValueError, AttributeError) as e:
    print('Error:', e)

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')